# Clustering — Variation 1: Daily Feature Clustering

Groups patient-days into interpretable clusters using engineered daily features: sensor-only features (steps, sleep, HR).  

## Setup & Data Loading

In [25]:
import sys, os, warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.decomposition import PCA
warnings.filterwarnings('ignore')
pd.set_option("display.max_columns", None)

# ── Path setup ────────────────────────────────────────────────────────────────
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..') if os.getcwd().endswith('notebooks') else os.getcwd())
sys.path.insert(0, PROJECT_ROOT)

import data_prep as dp
import clustering as cl

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
EXT_DIR  = os.path.join(DATA_DIR, 'external')

RANDOM_SEED = 72  # change here to re-run with a different seed

DISEASE_ORDER   = ['Early Disease Stage', 'Fast Disease Progression', 'Late Disease Stage']
DISEASE_PALETTE = {'Early Disease Stage': '#2ecc71', 'Fast Disease Progression': '#e74c3c', 'Late Disease Stage': '#3498db'}
CLUSTER_COLORS  = px.colors.qualitative.Plotly


In [26]:
import glob

files = glob.glob(os.path.join(DATA_DIR, 'Hourly Sensor Data', 'RHourly_*.csv'))
chunks = []
for fp in files:
    pid = int(os.path.basename(fp).replace('RHourly_', '').replace('.csv', ''))
    c = pd.read_csv(fp); c['id'] = pid; chunks.append(c)
sensor = pd.concat(chunks, ignore_index=True)
sensor['time']      = pd.to_datetime(sensor['time'])
sensor['heartrate'] = pd.to_numeric(sensor['heartrate'], errors='coerce')

clinical = pd.read_csv(os.path.join(DATA_DIR, 'ClinicalMarkers_final.csv'))
clinical.columns = clinical.columns.str.strip().str.lower().str.replace('.', '_', regex=False)
clinical['disease_type'] = clinical['disease_type'].str.strip()
df_merged = clinical[['id', 'age', 'sex', 'disease_type']].merge(sensor, on='id', how='left')

df = dp.get_complete_patient_days(df_merged, sleep_shift_hour=6, complete=True)
print(f'Complete rows: {len(df):,}  |  patients: {df["id"].nunique()}')


Complete rows: 44,949  |  patients: 43


In [27]:
df

,id,age,sex,disease_type,time,steps,sleep,heartrate,shifted_sleep,shifted_heartrate,n_complete_days
35207,1120,1982,Female,Late Disease Stage,2021-06-09 00:00:00,0.00000,0,72.097265,0.0,96.130433,52
35208,1120,1982,Female,Late Disease Stage,2021-06-09 01:00:00,0.00000,0,76.176299,0.0,93.885267,52
35209,1120,1982,Female,Late Disease Stage,2021-06-09 02:00:00,0.00000,0,78.143429,0.0,91.019185,52
35210,1120,1982,Female,Late Disease Stage,2021-06-09 03:00:00,0.00000,45,76.017211,0.0,78.088243,52
35211,1120,1982,Female,Late Disease Stage,2021-06-09 04:00:00,0.00000,60,73.999163,0.0,71.548728,52
...,...,...,...,...,...,...,...,...,...,...,...
24677,9926,1985,Female,Late Disease Stage,2021-06-07 17:00:00,4055.95700,0,107.751611,0.0,79.194127,38
24678,9926,1985,Female,Late Disease Stage,2021-06-07 18:00:00,253.45140,0,92.062437,0.0,85.443368,38
24679,9926,1985,Female,Late Disease Stage,2021-06-07 19:00:00,251.89790,0,80.565961,0.0,97.642828,38
24680,9926,1985,Female,Late Disease Stage,2021-06-07 20:00:00,131.90584,0,85.548315,0.0,88.280688,38


## Create features
- aggregate into daily level
- HR related features: z-score (taking into account individual baseline)
- The rest: standard scaler across all patients

In [28]:
daily = dp.aggregate_to_daily(df)
print(f'Daily shape: {daily.shape}')


Computing per-day features...
Daily shape: (1878, 29)


In [29]:
weather = dp.load_weather(os.path.join(EXT_DIR, 'weather', 'ogd-smn_klo_h_historical_2020-2029.csv'), daily=True)
cal     = dp.load_calendar(os.path.join(EXT_DIR, 'holidays', 'zurich_calendar_2021.csv'))
covid   = dp.load_covid(os.path.join(EXT_DIR, 'covid', 'ch_stringency_2021.csv'))
pollen  = dp.load_pollen(os.path.join(EXT_DIR, 'pollen', 'ogd-pollen_pzh_d_historical.csv'))

daily = daily.merge(weather, on='date', how='left')
daily = daily.merge(cal[['date', 'is_weekend', 'day_type']], on='date', how='left')
daily = daily.merge(covid[['date', 'stringency_index']], on='date', how='left')
daily = daily.merge(pollen[['date', 'pollen_total']], on='date', how='left')
# is_weekend as int for scaling
daily['is_weekend'] = daily['is_weekend'].astype(float)
print(f'Daily with external: {daily.shape}')

Daily with external: (1878, 39)


In [30]:
daily.describe()

,id,date,n_complete_days,age,daily_steps,active_hours,sedentary_hours,step_peak,peak_steps_hour,step_entropy,sum_sleep_minute,sleep_fragmentation,sleep_fragmentation_min,sleep_onset_hour,sleep_end_hour,mean_hr,max_hr,min_hr,resting_hr,day_hr,day_hr_var,resting_hr_var,hr_day_night_delta,ncc,ncc_per_step,no_active_hour,restless_night,temp_max,temp_mean,temp_min,precip_total,sunshine_total,humidity_mean,is_weekend,stringency_index,pollen_total
count,1878.000000,1878,1878.000000,1878.000000,1878.000000,1878.000000,1878.000000,1878.000000,1878.000000,1878.000000,1878.000000,1878.000000,1878.00000,1672.000000,1672.000000,1878.000000,1878.000000,1878.000000,1860.000000,1877.000000,1876.000000,1858.000000,1859.000000,1512.000000,1512.000000,1878.000000,1878.000000,1878.000000,1878.000000,1878.000000,1878.000000,1878.000000,1878.000000,1878.000000,1878.000000,1602.000000
mean,5639.051118,2021-06-19 11:31:37.763578368,46.676251,1977.449414,7601.766325,17.900958,6.033546,1739.932283,12.748136,2.400915,480.720980,2.240682,101.57934,4.192584,7.214115,78.599173,98.701398,62.875333,67.964037,84.686029,7.504177,3.803552,16.636935,23.826622,0.021154,0.187433,0.009585,17.267306,11.673234,6.314963,3.028807,335.561768,76.003352,0.283280,51.439398,63.315231
min,1120.000000,2021-01-09 00:00:00,1.000000,1958.000000,137.064649,3.000000,0.000000,80.016792,0.000000,0.943807,0.000000,0.000000,0.00000,0.000000,0.000000,62.140447,72.968372,45.463492,51.263887,64.127513,1.295683,0.310653,-8.243916,0.836482,0.000987,0.000000,0.000000,-5.200000,-6.866667,-14.800000,0.000000,0.000000,43.537500,0.000000,44.440000,0.000000
25%,3389.000000,2021-04-13 00:00:00,43.000000,1972.000000,3734.033102,16.000000,5.000000,731.374892,9.250000,2.268770,431.000000,2.000000,42.00000,3.000000,6.000000,73.079234,90.901838,57.665676,62.401250,78.752586,5.632969,2.439840,12.042396,18.869552,0.015461,0.000000,0.000000,12.500000,6.712500,1.100000,0.000000,95.000000,67.906250,0.000000,48.150000,4.000000
50%,5977.000000,2021-06-12 00:00:00,47.000000,1978.000000,6774.339522,18.000000,6.000000,1397.126700,13.000000,2.446751,490.000000,2.000000,63.00000,4.000000,6.000000,77.511473,97.686515,61.590277,66.685083,83.436826,7.130557,3.349435,16.707609,23.466540,0.019955,0.000000,0.000000,17.800000,11.550000,6.600000,0.000000,286.000000,77.487500,0.000000,50.000000,19.000000
75%,7513.000000,2021-09-05 00:00:00,52.000000,1982.000000,10496.751045,19.000000,7.000000,2317.694550,15.000000,2.572112,546.750000,2.000000,90.00000,5.000000,6.000000,82.179802,105.658444,66.963299,72.601250,89.195954,8.986609,4.725613,21.163604,28.464782,0.025697,0.000000,0.000000,22.600000,16.970833,11.900000,3.100000,541.000000,85.075000,1.000000,60.190000,84.000000
max,9926.000000,2021-11-14 00:00:00,71.000000,2005.000000,29545.063499,24.000000,18.000000,6088.826200,23.000000,2.951835,1042.000000,7.000000,775.00000,23.000000,23.000000,110.182995,139.439355,97.115911,100.496115,117.672698,23.043989,13.827582,44.513107,53.742961,0.061908,1.000000,1.000000,31.300000,24.654167,18.400000,60.300000,892.000000,99.275000,1.000000,60.190000,848.000000
std,2501.361895,NaN,8.504196,8.701977,4830.970323,2.267368,2.163396,1310.210219,3.878954,0.244125,116.632102,0.756859,128.76118,2.126461,4.219965,8.193122,10.876358,7.635156,8.060014,9.129247,2.662882,1.934497,7.282016,7.342083,0.008417,0.390363,0.097457,7.039583,6.275620,6.333569,6.923360,266.310815,11.257793,0.450711,5.763904,107.395426


In [31]:
daily = daily[daily['sum_sleep_minute']>0]  # Exclude days with no sleep data (likely device not worn)
feat_df = dp.build_daily_features(daily, include_external=False)
feature_cols = [c for c in feat_df.columns if c.endswith(('_sc', '_z', '_bool'))]
keep_cols    = ['id', 'date'] + [c for c in ['disease_type', 'sex', 'age'] if c in feat_df.columns]
print(f'Before imputation: {feat_df[feature_cols].isna().sum().sum()} NaN values')
feat_df[feature_cols].isna().sum()[lambda s: s > 0]

Before imputation: 714 NaN values


resting_hr_z              2
day_hr_z                  2
day_hr_var_z              3
resting_hr_var_z          4
hr_day_night_delta_z      3
ncc_z                   350
ncc_per_step_z          350
dtype: int64

## Impute missing values

In [32]:
# ── Impute NaN z-scores ───────────────────────────────────────────────────────
feat_df = feat_df.sort_values(['id', 'date']).copy()

# ncc_z, ncc_per_step_z → 2: sedentary days have no active HR signal;
# setting to +2 std places them distinctly above the average active day
for col in ['ncc_z', 'ncc_per_step_z']:
    if col in feat_df.columns:
        feat_df[col] = feat_df[col].fillna(2)

# resting/delta HR → carry last known value forward per patient;
# sleep HR baseline is stable day-to-day
for col in ['resting_hr_z', 'resting_hr_var_z', 'hr_day_night_delta_z']:
    if col in feat_df.columns:
        feat_df[col] = feat_df.groupby('id')[col].ffill()
        feat_df[col] = feat_df.groupby('id')[col].bfill()

# day_hr_z, day_hr_var_z → 0: missing daytime HR treated as typical
for col in ['day_hr_z', 'day_hr_var_z']:
    if col in feat_df.columns:
        feat_df[col] = feat_df[col].fillna(0)

print(f'After imputation:  {feat_df[feature_cols].isna().sum().sum()} NaN values remaining')

# ── Extract meta + feature matrix ─────────────────────────────────────────────
feat_df_clean = feat_df.dropna(subset=feature_cols).reset_index(drop=True)
meta = feat_df_clean[keep_cols]
X    = feat_df_clean[feature_cols].to_numpy(dtype=float)
print(f'Feature matrix: {X.shape}  ({len(feat_df) - len(feat_df_clean)} rows dropped)')
print(f'Features: {feature_cols}')

After imputation:  3 NaN values remaining
Feature matrix: (1860, 15)  (1 rows dropped)
Features: ['resting_hr_z', 'day_hr_z', 'day_hr_var_z', 'resting_hr_var_z', 'hr_day_night_delta_z', 'ncc_z', 'ncc_per_step_z', 'no_active_hour_bool', 'restless_night_bool', 'daily_steps_sc', 'active_hours_sc', 'step_peak_sc', 'step_entropy_sc', 'sum_sleep_minute_sc', 'sleep_fragmentation_min_sc']


In [33]:
feat_df[keep_cols+feature_cols].describe()

,id,date,age,resting_hr_z,day_hr_z,day_hr_var_z,resting_hr_var_z,hr_day_night_delta_z,ncc_z,ncc_per_step_z,no_active_hour_bool,restless_night_bool,daily_steps_sc,active_hours_sc,step_peak_sc,step_entropy_sc,sum_sleep_minute_sc,sleep_fragmentation_min_sc
count,1861.000000,1861,1861.000000,1860.000000,1.861000e+03,1.861000e+03,1860.000000,1860.000000,1861.000000,1861.000000,1861.000000,1861.000000,1.861000e+03,1.861000e+03,1.861000e+03,1.861000e+03,1.861000e+03,1.861000e+03
mean,5662.377217,2021-06-19 10:34:29.854916608,1977.408920,0.000446,2.916051e-16,2.099938e-17,-0.000287,-0.001403,0.376142,0.376142,0.186996,0.000537,1.374505e-16,-9.163367e-17,1.145421e-17,-1.359233e-15,-2.138119e-16,5.345297e-17
min,1120.000000,2021-01-09 00:00:00,1958.000000,-3.045448,-3.377461e+00,-2.634960e+00,-2.246641,-4.439812,-3.149685,-2.736727,0.000000,0.000000,-1.552306e+00,-6.587063e+00,-1.269644e+00,-5.963607e+00,-4.301798e+00,-7.949676e-01
25%,3389.000000,2021-04-13 00:00:00,1972.000000,-0.706846,-6.887357e-01,-7.074488e-01,-0.705975,-0.680532,-0.487104,-0.522692,0.000000,0.000000,-8.043830e-01,-8.379085e-01,-7.715838e-01,-5.425008e-01,-4.747764e-01,-4.614926e-01
50%,5977.000000,2021-06-12 00:00:00,1978.000000,-0.084686,-4.703663e-02,-7.192272e-02,-0.152248,0.029362,0.304145,0.165948,0.000000,0.000000,-1.724649e-01,4.657687e-02,-2.627231e-01,1.760142e-01,5.469013e-02,-2.986327e-01
75%,7996.000000,2021-09-06 00:00:00,1982.000000,0.562405,6.587432e-01,6.325418e-01,0.551731,0.687260,1.411411,1.514273,0.000000,0.000000,6.029262e-01,4.888195e-01,4.479618e-01,7.025633e-01,5.748678e-01,-9.699666e-02
max,9926.000000,2021-11-14 00:00:00,2005.000000,5.125885,4.493316e+00,4.277359e+00,4.897635,3.929695,3.234149,4.327384,1.000000,1.000000,4.564028e+00,2.700033e+00,3.323081e+00,2.257729e+00,5.172866e+00,5.215337e+00
std,2492.857494,NaN,8.701496,0.988826,9.883733e-01,9.881013e-01,0.988155,0.989501,1.183602,1.183602,0.390013,0.023181,1.000269e+00,1.000269e+00,1.000269e+00,1.000269e+00,1.000269e+00,1.000269e+00


## Fit model
- K-mean

In [34]:
SELECTED_FEATS = [
    'resting_hr_z',
    'day_hr_z',
    # 'day_hr_var_z',
    'resting_hr_var_z',
    'hr_day_night_delta_z',
    'ncc_z',
    # 'ncc_per_step_z',
    'no_active_hour_bool',
    # 'restless_night_bool',
    'daily_steps_sc',
    # 'active_hours_sc',
    'step_peak_sc',
    # 'step_entropy_sc',
    'sum_sleep_minute_sc',
    'sleep_fragmentation_min_sc'
    ]

In [35]:
# Filter to selected features only
feat_idx = [feature_cols.index(f) for f in SELECTED_FEATS if f in feature_cols]
X_r1    = X[:, feat_idx]
feat_r1 = [feature_cols[i] for i in feat_idx]

print(f'Full feature matrix: {X.shape}')
print(f'Selected feature matrix: {X_r1.shape}')
print(f'Selected features: {feat_r1}')
print(f'Rows dropped (NaN): {len(daily) - len(meta)}')


Full feature matrix: (1860, 15)
Selected feature matrix: (1860, 10)
Selected features: ['resting_hr_z', 'day_hr_z', 'resting_hr_var_z', 'hr_day_night_delta_z', 'ncc_z', 'no_active_hour_bool', 'daily_steps_sc', 'step_peak_sc', 'sum_sleep_minute_sc', 'sleep_fragmentation_min_sc']
Rows dropped (NaN): 1


In [36]:
print('Evaluating k=2..10 (Round 1)...')
eval_r1 = cl.evaluate_k(X_r1, range(2, 11), n_init=10, random_state=RANDOM_SEED)

fig = make_subplots(rows=1, cols=2,
    subplot_titles=['Elbow — Inertia', 'Silhouette Score (higher = better)'])
fig.add_trace(go.Scatter(x=eval_r1['k'], y=eval_r1['inertia'],
    mode='lines+markers', name='Inertia', line=dict(color='#3498db', width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=eval_r1['k'], y=eval_r1['silhouette'],
    mode='lines+markers', name='Silhouette', line=dict(color='#e74c3c', width=2)), row=1, col=2)
fig.update_xaxes(title_text='k (clusters)', dtick=1)
fig.update_layout(title='Cluster Count Evaluation',
    height=380, width=900, showlegend=False)
fig.show()
eval_r1

Evaluating k=2..10 (Round 1)...
  k=2: inertia=14575.1, silhouette=0.1593
  k=3: inertia=12242.8, silhouette=0.1940
  k=4: inertia=11008.2, silhouette=0.2022
  k=5: inertia=9965.9, silhouette=0.1789
  k=6: inertia=9220.5, silhouette=0.1708
  k=7: inertia=8834.6, silhouette=0.1520
  k=8: inertia=8492.7, silhouette=0.1506
  k=9: inertia=8205.9, silhouette=0.1529
  k=10: inertia=7929.6, silhouette=0.1477


,k,inertia,silhouette
0,2,14575.136483,0.159260
1,3,12242.838828,0.194008
2,4,11008.170309,0.202205
3,5,9965.880136,0.178893
4,6,9220.503779,0.170800
5,7,8834.616283,0.151991
6,8,8492.727539,0.150583
7,9,8205.876006,0.152927
8,10,7929.648783,0.147685


In [37]:
# ── Set k after inspecting the elbow / silhouette curves above ────────────────
K_R1 = 4  # adjust based on the plots

labels_r1, model_r1 = cl.fit_kmeans(X_r1, k=K_R1, random_state=RANDOM_SEED)
meta['cluster'] = labels_r1
print(f'Cluster sizes:\n{pd.Series(labels_r1).value_counts().sort_index().to_string()}')

KMeans k=4: inertia=11008.2, silhouette=0.2022
Cluster sizes:
0    703
1    631
2    383
3    143


In [38]:
# ---- Feature importance: one-way ANOVA F-statistic across clusters ----
# High F = this feature differs most between clusters (drives separation).
feat_imp = cl.feature_importance_anova(X_r1, labels_r1, feat_r1)

fig = px.bar(
    feat_imp, x='F_stat', y='raw_feature', orientation='h',
    color='F_stat', color_continuous_scale='Reds',
    error_x=None,
    hover_data={'feature': True, 'p_value': ':.4f'},
    labels={'raw_feature': 'Feature', 'F_stat': 'ANOVA F-statistic'},
    title='Round 1 — Feature Importance: Which features drive cluster separation?',
    width=700, height=420,
)
fig.update_layout(yaxis=dict(categoryorder='total ascending'), coloraxis_showscale=False)
fig.show()
feat_imp[['raw_feature', 'feature', 'F_stat', 'p_value']].round(3)


,raw_feature,feature,F_stat,p_value
0,sleep_fragmentation_min,sleep_fragmentation_min_sc,2782.826,0.0
1,no_active_hour,no_active_hour_bool,1158.791,0.0
2,ncc,ncc_z,1020.133,0.0
3,daily_steps,daily_steps_sc,510.083,0.0
4,hr_day_night_delta,hr_day_night_delta_z,449.884,0.0
5,step_peak,step_peak_sc,401.420,0.0
6,day_hr,day_hr_z,218.797,0.0
7,resting_hr,resting_hr_z,81.749,0.0
8,resting_hr_var,resting_hr_var_z,43.824,0.0
9,sum_sleep_minute,sum_sleep_minute_sc,40.379,0.0


## Visualization

In [39]:
pca = PCA(n_components=2, random_state=RANDOM_SEED)
Z   = pca.fit_transform(X_r1)
pca_df = meta.copy()
pca_df['PC1'] = Z[:, 0]
pca_df['PC2'] = Z[:, 1]
pca_df['cluster_str'] = 'Cluster ' + pca_df['cluster'].astype(str)

fig = px.scatter(
    pca_df, x='PC1', y='PC2',
    color='cluster_str',
    symbol='disease_type' if 'disease_type' in pca_df.columns else None,
    hover_data={'id': True, 'date': '|%Y-%m-%d', 'cluster': True,
                'disease_type': True} if 'disease_type' in pca_df.columns else {'id': True},
    title=f'PCA (2D): {K_R1} clusters, sensor-only features',
    labels={'cluster_str': 'Cluster', 'PC1': f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)',
            'PC2': f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)'},
    width=820, height=520,
)
fig.show()

In [40]:
# Summary table — raw feature means per cluster
# (desc_r1 stores raw column names; feature_cols has _sc/_z suffixes — don't mix them)
desc_r1 = cl.describe_clusters(meta, labels_r1, daily, feat_r1)
mean_cols    = [col for col in desc_r1.columns if col.startswith('mean_')]
disease_cols = [col for col in ['pct_early', 'pct_fast', 'pct_late'] if col in desc_r1.columns]
display_cols = ['n_days', 'pct_total'] + mean_cols + disease_cols
desc_r1[display_cols].round(2)


,n_days,pct_total,mean_resting_hr,mean_day_hr,mean_resting_hr_var,mean_hr_day_night_delta,mean_ncc,mean_no_active_hour,mean_daily_steps,mean_step_peak,mean_sum_sleep_minute,mean_sleep_fragmentation_min,pct_early,pct_fast,pct_late
cluster,,,,,,,,,,,,,,,
0,703,37.80,69.09,83.65,3.97,14.56,20.23,0.00,7077.63,1556.42,485.77,68.83,37.27,16.50,46.23
1,631,33.92,67.10,89.18,3.19,22.08,27.52,0.00,11557.40,2765.12,470.33,67.19,46.75,16.16,37.08
2,383,20.59,66.01,79.93,3.89,13.90,30.68,0.78,2835.85,562.86,474.90,71.68,23.76,15.67,60.57
3,143,7.69,71.55,81.56,5.48,10.02,19.87,0.34,5442.83,1264.88,574.53,506.73,16.08,16.78,67.13


In [41]:
# Centroid z-score heatmap — how each cluster deviates from the cross-cluster average
# Each feature is standardised across clusters so all features sit on the same scale.
# Red = this cluster is high on this feature; blue = low.
mean_cols = [c for c in desc_r1.columns if c.startswith('mean_')]
raw_names = [c.replace('mean_', '') for c in mean_cols]

centroid_vals = desc_r1[mean_cols].copy()
centroid_vals.columns = raw_names

centroid_z = (centroid_vals - centroid_vals.mean()) / centroid_vals.std().replace(0, 1)

short_name = {
    'daily_steps':              'Steps/day',
    'active_hours':             'Active hrs',
    'step_peak':                'Peak steps',
    'step_entropy':             'Step entropy',
    'sum_sleep_minute':         'Sleep (min)',
    'sleep_fragmentation_min':  'Sleep frag.',
    'no_active_hour':           'No vigorous act.',
    'restless_night':           'Restless night',
    'resting_hr':               'Resting HR',
    'day_hr':                   'Day HR',
    'day_hr_var':               'Day HR var',
    'resting_hr_var':           'Resting HR var',
    'hr_day_night_delta':       'HR Δ day–night',
    'ncc':                      'NCC',
    'ncc_per_step':             'NCC / step',
}
x_labels = [short_name.get(c, c) for c in centroid_z.columns]
y_labels  = [f'C{i}  (n={int(desc_r1.loc[i, "n_days"])})' for i in centroid_z.index]

fig = go.Figure(go.Heatmap(
    z=centroid_z.values,
    x=x_labels,
    y=y_labels,
    colorscale='RdBu_r',
    zmid=0,
    text=centroid_z.round(1).astype(str).values,
    texttemplate='%{text}',
    textfont=dict(size=11),
    colorbar=dict(title='z-score<br>across<br>clusters', thickness=14),
    hovertemplate='%{y} — %{x}<br>z = %{z:.2f}<extra></extra>',
))
fig.update_layout(
    title='Cluster Centroid Profiles — z-scored across clusters<br>'
          '<sup>Red = above cluster average · Blue = below · Values in SD units</sup>',
    height=360, width=980,
    xaxis=dict(tickangle=-35, side='bottom'),
    yaxis=dict(autorange='reversed'),
)
fig.show()

### Cluster interpretation
1. C0: normal day with light activity
2. C1: very active day with lots of steps (with good resting HR the night before)
3. C2: well rested but inactive day 
4. C3: lots of time in bed + fragmented sleep patterns

## Feature importance

In [47]:
# Cluster composition: x = disease stage, y = patient-days, colour = cluster
if 'disease_type' in meta.columns:
    comp = meta.groupby(['disease_type', 'cluster']).size().reset_index(name='n')
    comp['cluster_str'] = 'Cluster ' + comp['cluster'].astype(str)

    # Count chart
    fig = px.bar(
        comp, x='disease_type', y='n', color='cluster_str',
        color_discrete_sequence=CLUSTER_COLORS,
        category_orders={'disease_type': DISEASE_ORDER},
        title='Patient-days per Disease Stage (count)',
        labels={'disease_type': 'Disease Stage', 'n': 'Patient-days', 'cluster_str': 'Cluster'},
        barmode='stack', width=720, height=420,
    )
    fig.show()

    # Percentage chart — within each disease stage, what % of days per cluster?
    total_per_stage = comp.groupby('disease_type')['n'].transform('sum')
    comp['pct'] = comp['n'] / total_per_stage * 100

    fig2 = px.bar(
        comp, x='disease_type', y='pct', color='cluster_str',
        color_discrete_sequence=CLUSTER_COLORS,
        category_orders={'disease_type': DISEASE_ORDER},
        title='Cluster Distribution within each Disease Stage (%)',
        labels={'disease_type': 'Disease Stage', 'pct': '% of days in stage', 'cluster_str': 'Cluster'},
        barmode='stack', width=720, height=420,
        text=comp['pct'].round(1).astype(str) + '%',
    )
    fig2.update_traces(textposition='inside', textfont_size=11)
    fig2.update_layout(yaxis_range=[0, 100])
    fig2.show()


In [48]:
# Calendar heatmap: patient x date, discrete colour per cluster
cal_df = meta.copy()
if 'disease_type' not in cal_df.columns:
    cal_df = cal_df.merge(daily[['id','date','disease_type']], on=['id','date'], how='left')

# Rank: disease stage (Early → Fast → Late) then first complete date within stage
_stage_rank   = {s: i for i, s in enumerate(DISEASE_ORDER)}
_patient_info = (
    cal_df.groupby('id')
    .agg(disease_type=('disease_type', 'first'),
         first_date=('date', 'min'))
    .reset_index()
    .assign(stage_rank=lambda d: d['disease_type'].map(_stage_rank))
    .sort_values(['stage_rank', 'first_date'])
)
patient_order = _patient_info['id'].tolist()

pivot = cal_df.pivot(index='id', columns='date', values='cluster').reindex(patient_order)

# Step-function colorscale: each cluster band maps to the same colour as disease-bar / relative-day
colorscale = []
for i in range(K_R1):
    lo = i / K_R1
    hi = (i + 1) / K_R1
    colorscale.append([lo, CLUSTER_COLORS[i % len(CLUSTER_COLORS)]])
    colorscale.append([hi, CLUSTER_COLORS[i % len(CLUSTER_COLORS)]])

# y-axis labels: "id (stage abbr.)"
_stage_abbr = {'Early Disease Stage': 'Early', 'Fast Disease Progression': 'Fast', 'Late Disease Stage': 'Late'}
_id_to_stage = _patient_info.set_index('id')['disease_type']
y_labels = [f'{pid} ({_stage_abbr.get(_id_to_stage[pid], "?")})' for pid in patient_order]

fig = go.Figure(go.Heatmap(
    z=pivot.values,
    x=[str(d)[:10] for d in pivot.columns],
    y=y_labels,
    colorscale=colorscale,
    zmin=-0.5, zmax=K_R1 - 0.5,
    colorbar=dict(
        title='Cluster',
        tickvals=list(range(K_R1)),
        ticktext=[f'Cluster {k}' for k in range(K_R1)],
        lenmode='fraction', len=0.6,
    ),
    hovertemplate='Date: %{x}<br>Patient: %{y}<br>Cluster: %{z}<extra></extra>',
))

# Horizontal dividers between disease stage groups
_stage_sizes = _patient_info.groupby('stage_rank')['id'].count().sort_index().tolist()
_shapes, _annotations = [], []
_cum = 0
for stage, sz in zip(DISEASE_ORDER, _stage_sizes):
    _annotations.append(dict(
        x=1.08, xref='paper', y=_cum + sz / 2,
        text=_stage_abbr[stage], showarrow=False,
        font=dict(size=11), xanchor='left',
    ))
    _cum += sz
    if _cum < len(patient_order):
        _shapes.append(dict(
            type='line', x0=0, x1=1, xref='paper',
            y0=_cum - 0.5, y1=_cum - 0.5,
            line=dict(color='black', width=1.5, dash='dot'),
        ))

fig.update_layout(
    title='Calendar Heatmap: Cluster Assignment per Patient-day<br>'
          '<sup>Sorted by disease stage then first study date</sup>',
    height=750, width=1150,
    shapes=_shapes,
    annotations=_annotations,
    xaxis=dict(title='Date', tickangle=-45),
    yaxis=dict(title='Patient ID', autorange='reversed', tickfont=dict(size=9)),
)
fig.show()

In [49]:
# Relative-day coverage grid coloured by cluster
# x = study day relative to each patient's first complete day
# y = patient, ordered by disease stage then total days (descending)

_stage_abbr = {
    'Early Disease Stage':      'Early',
    'Fast Disease Progression': 'Fast',
    'Late Disease Stage':       'Late',
}

_first = meta.groupby('id')['date'].min().rename('first_date').reset_index()
meta_rd = meta.merge(_first, on='id')
meta_rd['relative_day'] = (meta_rd['date'] - meta_rd['first_date']).dt.days + 1

patient_info = (
    meta_rd.groupby('id')
    .agg(disease_type=('disease_type', 'first'), n_days=('date', 'nunique'))
    .reset_index()
)
_stage_rank = {s: i for i, s in enumerate(DISEASE_ORDER)}
_id_order = (
    patient_info
    .assign(_r=patient_info['disease_type'].map(_stage_rank))
    .sort_values(['_r', 'n_days'], ascending=[True, False])['id']
    .tolist()
)
_y_pos    = {pid: i for i, pid in enumerate(_id_order)}
_y_labels = [
    f'{pid} ({_stage_abbr.get(patient_info.loc[patient_info["id"]==pid, "disease_type"].values[0], "?")})'
    for pid in _id_order
]

meta_rd['y_pos'] = meta_rd['id'].map(_y_pos)

fig = go.Figure()
for k in range(K_R1):
    sub = meta_rd[meta_rd['cluster'] == k]
    fig.add_trace(go.Scatter(
        x=sub['relative_day'],
        y=sub['y_pos'],
        mode='markers',
        marker=dict(symbol='square', size=8, color=CLUSTER_COLORS[k % len(CLUSTER_COLORS)]),
        name=f'Cluster {k}',
        customdata=sub[['id', 'disease_type']].values,
        hovertemplate=(
            'Patient %{customdata[0]}<br>'
            'Relative day: %{x}<br>'
            f'Cluster: {k}<br>'
            'Stage: %{customdata[1]}'
            '<extra></extra>'
        ),
    ))

# Horizontal dividers between disease stage groups
_stage_sizes = [
    sum(1 for pid in _id_order
        if patient_info.loc[patient_info['id'] == pid, 'disease_type'].values[0] == s)
    for s in DISEASE_ORDER
]
_cum = 0
_shapes, _annotations = [], []
for stage, sz in zip(DISEASE_ORDER, _stage_sizes):
    _annotations.append(dict(
        x=1.01, xref='paper', y=_cum + sz / 2,
        text=_stage_abbr[stage], showarrow=False,
        font=dict(size=10), xanchor='left',
    ))
    _cum += sz
    if _cum < len(_id_order):  # no divider after last group
        _shapes.append(dict(
            type='line', x0=0, x1=1, xref='paper',
            y0=_cum - 0.5, y1=_cum - 0.5,
            line=dict(color='black', width=1, dash='dot'),
        ))

fig.update_layout(
    title='Relative Study-Day Coverage (coloured by cluster)',
    height=700, width=1000,
    shapes=_shapes,
    annotations=_annotations,
    xaxis=dict(title='Relative study day', dtick=7),
    yaxis=dict(
        tickvals=list(range(len(_id_order))),
        ticktext=_y_labels,
        autorange='reversed',
        tickfont=dict(size=9),
    ),
)
fig.show()


In [50]:
from sklearn.metrics import pairwise_distances_argmin

centroid_idx = pairwise_distances_argmin(model_r1.cluster_centers_, X_r1)

# Pre-fetch representative days
rep_days = []
for k in range(K_R1):
    row_meta = meta.iloc[centroid_idx[k]]
    pid, day = row_meta['id'], row_meta['date']
    day_df = (
        df[(df['id'] == pid) & (df['time'].dt.normalize() == pd.Timestamp(day))]
        .sort_values('time')
        .copy()
    )
    rep_days.append((k, pid, day, day_df))

# Universal axis ranges across all clusters
max_steps = max(d['steps'].max() for _, _, _, d in rep_days)
hr_all    = pd.concat([d['heartrate'].dropna() for _, _, _, d in rep_days])
hr_range  = [hr_all.min() * 0.95, hr_all.max() * 1.05]

top_titles = [f'Cluster {k}' for k, _, _, _ in rep_days]
bot_titles = [f'P{pid}  {str(day)[:10]}' for _, pid, day, _ in rep_days]

specs = [[{'secondary_y': True}] * K_R1, [{'secondary_y': False}] * K_R1]
fig = make_subplots(
    rows=2, cols=K_R1,
    subplot_titles=top_titles + bot_titles,
    specs=specs,
    vertical_spacing=0.18, horizontal_spacing=0.08,
)

for k, pid, day, day_df in rep_days:
    clock_labels       = [f"{t.hour:02d}:00" for t in day_df['time']]
    clock_labels_sleep = [f"{(t.hour - 6) % 24:02d}:00" for t in day_df['time']]
    col = k + 1
    show_legend = (k == 0)

    fig.add_trace(go.Bar(
        x=clock_labels, y=day_df['steps'],
        name='Steps/hr', marker_color='#3498db',
        showlegend=show_legend,
    ), row=1, col=col, secondary_y=False)

    fig.add_trace(go.Scatter(
        x=clock_labels, y=day_df['heartrate'],
        mode='lines+markers', marker=dict(size=3),
        name='HR (bpm)', line=dict(color='#e74c3c', width=1.5),
        showlegend=show_legend,
    ), row=1, col=col, secondary_y=True)

    fig.add_trace(go.Bar(
        x=clock_labels_sleep, y=day_df['shifted_sleep'],
        name='Sleep min/hr', marker_color='#9b59b6',
        showlegend=show_legend,
    ), row=2, col=col)

    # Universal ranges
    fig.update_yaxes(range=[0, max_steps * 1.05], row=1, col=col, secondary_y=False)
    fig.update_yaxes(range=hr_range, row=1, col=col, secondary_y=True)

    for r in (1, 2):
        fig.update_xaxes(tickangle=-60, tickfont_size=8, row=r, col=col)

fig.update_yaxes(title_text='Steps / hr', row=1, col=1, secondary_y=False)
fig.update_yaxes(title_text='HR (bpm)', row=1, col=K_R1, secondary_y=True)
fig.update_yaxes(title_text='Sleep min / hr', row=2, col=1)

fig.update_layout(
    title='Representative Day per Cluster (closest to centroid)',
    height=520, width=max(900, 280 * K_R1),
    bargap=0.05,
)
fig.show()


In [51]:
from sklearn.metrics import pairwise_distances

N_EXAMPLES = 3  # examples per cluster

# ── N closest days to centroid per cluster ────────────────────────────────────
cluster_examples = {}
for k in range(K_R1):
    mask  = np.where(labels_r1 == k)[0]
    dists = pairwise_distances(model_r1.cluster_centers_[[k]], X_r1[mask])[0]
    cluster_examples[k] = mask[np.argsort(dists)[:N_EXAMPLES]]

# ── Pre-fetch hourly DataFrames ───────────────────────────────────────────────
example_days = {}
for k, idxs in cluster_examples.items():
    days = []
    for idx in idxs:
        row = meta.iloc[idx]
        pid, day = row['id'], row['date']
        day_df = (
            df[(df['id'] == pid) & (df['time'].dt.normalize() == pd.Timestamp(day))]
            .sort_values('time').copy()
        )
        days.append((pid, day, day_df))
    example_days[k] = days

# ── Global axis ranges ────────────────────────────────────────────────────────
all_dfs   = [d for days in example_days.values() for _, _, d in days]
max_steps = max(d['steps'].max() for d in all_dfs)
hr_all    = pd.concat([d['heartrate'].dropna() for d in all_dfs])
hr_range  = [hr_all.min() * 0.95, hr_all.max() * 1.05]

# ── Layout: 2 rows per cluster group (steps/HR + sleep) × N_EXAMPLES cols ────
rows_per_cluster = 2
total_rows = K_R1 * rows_per_cluster

specs = []
for _ in range(K_R1):
    specs.append([{'secondary_y': True}]  * N_EXAMPLES)   # steps + HR
    specs.append([{'secondary_y': False}] * N_EXAMPLES)   # sleep

# Subplot titles: cluster label + patient info on steps/HR rows; blank on sleep rows
subplot_titles = []
for k in range(K_R1):
    for n, (pid, day, _) in enumerate(example_days[k]):
        prefix = f'<b>Cluster {k}</b> · ' if n == 0 else ''
        subplot_titles.append(f'{prefix}P{pid}  {str(day)[:10]}')
    subplot_titles += [''] * N_EXAMPLES   # sleep row

fig = make_subplots(
    rows=total_rows, cols=N_EXAMPLES,
    specs=specs,
    subplot_titles=subplot_titles,
    vertical_spacing=0.05,
    horizontal_spacing=0.06,
)

for k in range(K_R1):
    sig_row = k * rows_per_cluster + 1
    slp_row = sig_row + 1
    show_legend = (k == 0)

    for n, (pid, day, day_df) in enumerate(example_days[k]):
        col         = n + 1
        clock       = [f'{t.hour:02d}:00' for t in day_df['time']]
        clock_sleep = [f'{(t.hour - 6) % 24:02d}:00' for t in day_df['time']]

        fig.add_trace(go.Bar(
            x=clock, y=day_df['steps'],
            name='Steps/hr', marker_color='#3498db',
            showlegend=(show_legend and n == 0),
        ), row=sig_row, col=col, secondary_y=False)

        fig.add_trace(go.Scatter(
            x=clock, y=day_df['heartrate'],
            mode='lines+markers', marker=dict(size=3),
            name='HR (bpm)', line=dict(color='#e74c3c', width=1.5),
            showlegend=(show_legend and n == 0),
        ), row=sig_row, col=col, secondary_y=True)

        fig.add_trace(go.Bar(
            x=clock_sleep, y=day_df['shifted_sleep'],
            name='Sleep min/hr', marker_color='#9b59b6',
            showlegend=(show_legend and n == 0),
        ), row=slp_row, col=col)

        fig.update_yaxes(range=[0, max_steps * 1.05], row=sig_row, col=col, secondary_y=False)
        fig.update_yaxes(range=hr_range,              row=sig_row, col=col, secondary_y=True)
        fig.update_yaxes(range=[0, 65],               row=slp_row, col=col)
        for r in (sig_row, slp_row):
            fig.update_xaxes(tickangle=-60, tickfont_size=7, row=r, col=col)

    # y-axis labels on left column only
    fig.update_yaxes(title_text='Steps/hr',    row=sig_row, col=1, secondary_y=False, title_font_size=10)
    fig.update_yaxes(title_text='Sleep min/hr', row=slp_row, col=1, title_font_size=10)

fig.update_layout(
    title=f'Example Days per Cluster — {N_EXAMPLES} closest to centroid',
    height=240 * total_rows,
    width=max(850, 300 * N_EXAMPLES),
    bargap=0.05,
    legend=dict(orientation='h', y=1.01, x=0),
)
fig.show()

## Export Cluster Labels

In [87]:
label_df_r1 = cl.labels_to_dataframe(meta, labels_r1, label_col='cluster_v1')
label_df_r1.to_csv(os.path.join(DATA_DIR, 'cluster_labels_v1.csv'), index=False)
print(f'Saved cluster_labels_v1.csv: {label_df_r1.shape}')
label_df_r1.head()


Saved cluster_labels_v1.csv: (1915, 3)


,id,date,cluster_v1
0,1120,2021-06-09,3
1,1120,2021-06-10,0
2,1120,2021-06-11,3
3,1120,2021-06-13,0
4,1120,2021-06-14,0
